In [1]:
import IPython
IPython.display.display(IPython.display.Javascript('''
    function clickConnect() {
        var buttons = document.querySelectorAll("colab-connect-button");
        if (buttons.length > 0) buttons[0].click();
        setTimeout(clickConnect, 60000);
    }
    clickConnect();
'''))

<IPython.core.display.Javascript object>

# 🎙️ Experiment 1 — Combined ASCEND + Your Voice Fine-Tuning

Trains Whisper Tiny on a mixture of ASCEND Chinese-accented English and your own voice recordings.
Your voice is oversampled 3× so it isn't dominated by ASCEND's larger size.
Eval is on your voice only throughout.

### Workflow:
1. Check GPU
2. Install dependencies
3. Listen to ASCEND samples
4. Upload your dataset
5. Validate
6. Build combined dataset (ASCEND + your voice ×3)
7. Prepare features
8. Normalizer + collator + metrics
9. Experiment config
10. Run fine-tuning
11. Compare experiments
12. Final comparison — baseline vs fine-tuned on your val set
13. Test on different speaker clips
14. Download best model

---
**Runtime:** A100 GPU recommended — Runtime > Change runtime type > A100

## Step 0 — Check GPU

In [2]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else '❌ No GPU found. Go to Runtime > Change runtime type > A100 GPU')

Sat Mar  7 05:38:48 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P0             43W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## Step 1 — Install Dependencies

In [3]:
!apt-get install -q ffmpeg
!pip install -q transformers datasets librosa soundfile jiwer accelerate evaluate scikit-learn
print('✅ Dependencies installed')

Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 37 not upgraded.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 118.0 MB/s eta 0:00:00
✅ Dependencies installed


## Step 2 — Listen to ASCEND Samples

Before training, listen to a few clips to hear what the accent sounds like.

In [4]:
# Listen to a few ASCEND English samples to hear the accent
print('🎧 Playing 3 sample ASCEND English clips in notebook...')
from datasets import load_dataset
from IPython.display import Audio, display
import numpy as np

ascend = load_dataset('CAiRE/ASCEND', streaming=True, trust_remote_code=True)
count = 0
for sample in ascend['train']:
    if sample['language'] == 'en' and count < 3:
        print(f'\n  Transcript: "{sample["transcription"]}"')
        display(Audio(sample['audio']['array'], rate=sample['audio']['sampling_rate']))
        count += 1
    if count >= 3:
        break
print('\n✅ Done — these are the speakers your model is training on')

🎧 Playing 3 sample ASCEND English clips in notebook...


`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'CAiRE/ASCEND' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'CAiRE/ASCEND' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this sec

README.md: 0.00B [00:00, ?B/s]


  Transcript: "we have the sea shore in the city and we have a lot of"



  Transcript: "delicious sea food"



  Transcript: "um watch some movies"



✅ Done — these are the speakers your model is training on


## Step 3 — Upload Your Dataset

Upload the `dataset.zip` from `prepare_dataset.py` — contains `clips/` and `metadata.csv`.

In [5]:
from google.colab import files
import zipfile, os, pandas as pd

print('📁 Upload your dataset zip (dataset.zip containing clips/ and metadata.csv)...')
uploaded = files.upload()

zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content/')

print('✅ Files extracted')
!find /content/dataset -type f | head -20

📁 Upload your dataset zip (dataset.zip containing clips/ and metadata.csv)...


Saving dataset.zip to dataset.zip
✅ Files extracted
/content/dataset/clips/clip_0073.wav
/content/dataset/clips/clip_0153.wav
/content/dataset/clips/clip_0242.wav
/content/dataset/clips/clip_0091.wav
/content/dataset/clips/clip_0186.wav
/content/dataset/clips/clip_0087.wav
/content/dataset/clips/clip_0040.wav
/content/dataset/clips/clip_0008.wav
/content/dataset/clips/clip_0070.wav
/content/dataset/clips/clip_0152.wav
/content/dataset/clips/clip_0303.wav
/content/dataset/clips/clip_0018.wav
/content/dataset/clips/clip_0080.wav
/content/dataset/clips/clip_0074.wav
/content/dataset/clips/clip_0022.wav
/content/dataset/clips/clip_0039.wav
/content/dataset/clips/clip_0024.wav
/content/dataset/clips/clip_0287.wav
/content/dataset/clips/clip_0019.wav
/content/dataset/clips/clip_0151.wav


## Step 4 — Load Paths and Validate

In [6]:
import os, pandas as pd

DATA_DIR      = '/content/dataset'
CLIPS_DIR     = os.path.join(DATA_DIR, 'clips')
METADATA_PATH = os.path.join(DATA_DIR, 'metadata.csv')

df = pd.read_csv(METADATA_PATH)
print(f'📊 Loaded {len(df)} rows from metadata.csv')
print(df.head())

📊 Loaded 335 rows from metadata.csv
       file_name                                         transcript  \
0  clip_0001.wav                  alright guys let's get it started   
1  clip_0002.wav  today we are officially entering the world of ...   
2  clip_0003.wav  youve probably heard this term everywhere in j...   
3  clip_0004.wav  so today my goal is not to make this scary or ...   
4  clip_0005.wav       instead i want you to walk out understanding   

   duration_sec            source  
0          4.40  r1/clip_0001.wav  
1          3.50  r1/clip_0002.wav  
2         10.24  r1/clip_0003.wav  
3          4.19  r1/clip_0004.wav  
4          2.81  r1/clip_0005.wav  


In [7]:
import librosa

total_duration = 0
issues = []

for _, row in df.iterrows():
    fname = row['file_name']
    path  = os.path.join(CLIPS_DIR, fname)
    if not os.path.exists(path):
        issues.append(f'Missing: {fname}')
        continue
    try:
        duration = librosa.get_duration(path=path)
        total_duration += duration
        if duration > 30:
            issues.append(f'Too long ({duration:.1f}s): {fname}')
    except Exception as e:
        issues.append(f'Could not read {fname}: {e}')

print(f'⏱️  Total audio duration: {total_duration/60:.1f} minutes')
if issues:
    print('\n⚠️ Issues found:')
    for i in issues: print(f'  - {i}')
else:
    print('✅ All files look good!')

⏱️  Total audio duration: 44.1 minutes
✅ All files look good!


## Step 5 — Build Combined Dataset

Loads ASCEND English subset via streaming and combines with your voice.
Your voice is repeated 3× to balance the dataset sizes.

In [8]:
from datasets import Dataset, Audio, load_dataset
from sklearn.model_selection import train_test_split
import numpy as np, librosa, soundfile as sf

TARGET_SR = 16000

# ── Load your voice clips ─────────────────────────────────────────────────────
df = df.dropna(subset=['file_name', 'transcript'])
df['path'] = df['file_name'].apply(lambda f: os.path.join(CLIPS_DIR, f))
df = df[df['path'].apply(lambda x: isinstance(x, str))]
df = df[df['transcript'].apply(lambda x: isinstance(x, str) and x.strip() != '')]
df = df[df['path'].apply(os.path.exists)].reset_index(drop=True)
print(f'✅ Your voice clips: {len(df)}')

# ── Train/val split on your voice first ───────────────────────────────────────
train_df, val_df = train_test_split(df, test_size=0.15, random_state=42)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
print(f'   Train: {len(train_df)} | Val: {len(val_df)}')

# ── Load voice arrays into memory ─────────────────────────────────────────────
print('\n🔄 Loading your voice clips into memory...')
voice_arrays = []
voice_trans  = []
for _, row in train_df.iterrows():
    audio, _ = sf.read(row['path'])
    audio = audio.astype(np.float32)
    if audio.ndim > 1: audio = audio.mean(axis=1)
    voice_arrays.append(audio)
    voice_trans.append(row['transcript'])
print(f'✅ Loaded {len(voice_arrays)} voice clips')

# ── Load ASCEND English subset ─────────────────────────────────────────────────
print('\n🔄 Streaming ASCEND English samples...')
ascend_stream = load_dataset('CAiRE/ASCEND', streaming=True, trust_remote_code=True)
ascend_arrays = []
ascend_trans  = []
for sample in ascend_stream['train']:
    if sample['language'] == 'en':
        arr = sample['audio']['array'].astype(np.float32)
        sr  = sample['audio']['sampling_rate']
        if sr != TARGET_SR:
            arr = librosa.resample(arr, orig_sr=sr, target_sr=TARGET_SR)
        ascend_arrays.append(arr)
        ascend_trans.append(sample['transcription'])
print(f'✅ ASCEND English samples: {len(ascend_arrays)}')

# ── Combine: voice ×3 oversample + ASCEND ─────────────────────────────────────
OVERSAMPLE = 3
all_arrays = voice_arrays * OVERSAMPLE + ascend_arrays
all_trans  = voice_trans  * OVERSAMPLE + ascend_trans
print(f'\n📊 Combined train: {len(all_arrays)} samples')
print(f'   Your voice (×{OVERSAMPLE}): {len(voice_arrays) * OVERSAMPLE}')
print(f'   ASCEND: {len(ascend_arrays)}')

# ── Build single unified dataset from arrays ──────────────────────────────────
# Everything is a numpy array now — no type mismatch possible
import random
combined = list(zip(all_arrays, all_trans))
random.shuffle(combined)
all_arrays_shuffled, all_trans_shuffled = zip(*combined)

train_dataset = Dataset.from_dict({
    'audio':    [{'array': a, 'sampling_rate': TARGET_SR} for a in all_arrays_shuffled],
    'sentence': list(all_trans_shuffled)
})

val_dataset = Dataset.from_dict({
    'audio':    [{'array': a, 'sampling_rate': TARGET_SR} for a in
                 [sf.read(p)[0].astype(np.float32) for p in val_df['path']]],
    'sentence': val_df['transcript'].tolist()
})

print(f'\n✅ train_dataset: {len(train_dataset)}')
print(f'✅ val_dataset  : {len(val_dataset)}')

✅ Your voice clips: 332
   Train: 282 | Val: 50

🔄 Loading your voice clips into memory...


`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'CAiRE/ASCEND' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'CAiRE/ASCEND' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


✅ Loaded 282 voice clips

🔄 Streaming ASCEND English samples...
✅ ASCEND English samples: 2331

📊 Combined train: 3177 samples
   Your voice (×3): 846
   ASCEND: 2331

✅ train_dataset: 3177
✅ val_dataset  : 50


## Step 6 — Load Processor and Prepare Features

In [9]:
from transformers import WhisperProcessor

processor = WhisperProcessor.from_pretrained('openai/whisper-tiny.en')

def prepare_dataset(batch):
    audio = batch['audio']
    batch['input_features'] = processor(
        audio['array'],
        sampling_rate=audio['sampling_rate'],
        return_tensors='pt'
    ).input_features[0]
    batch['labels'] = processor.tokenizer(batch['sentence']).input_ids
    return batch

print('🔄 Preparing combined train features...')
train_dataset = train_dataset.map(prepare_dataset, remove_columns=train_dataset.column_names)
print('🔄 Preparing val features...')
val_dataset   = val_dataset.map(prepare_dataset, remove_columns=val_dataset.column_names)
print(f'✅ Train: {len(train_dataset)} | Val: {len(val_dataset)}')

preprocessor_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/805 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

🔄 Preparing combined train features...


Map:   0%|          | 0/3177 [00:00<?, ? examples/s]

🔄 Preparing val features...


Map:   0%|          | 0/50 [00:00<?, ? examples/s]

✅ Train: 3177 | Val: 50


## Step 7 — Text Normalizer, Collator and Metrics

In [10]:
import torch, evaluate, re, string
from dataclasses import dataclass
from typing import Any, Dict, List, Union

# ── Text normalizer ───────────────────────────────────────────────────────────
# Applied to both Whisper output and reference transcripts before WER.
# Whisper outputs "The weather is nice." — we normalize to "the weather is nice"
def normalize_text(text):
    text = text.lower()
    punct = string.punctuation.replace("'", "")
    text  = text.translate(str.maketrans('', '', punct))
    text  = re.sub(r'\s+', ' ', text).strip()
    return text

# ── Data collator ─────────────────────────────────────────────────────────────
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]
        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

# ── Metrics ───────────────────────────────────────────────────────────────────
metric = evaluate.load('wer')

def compute_metrics(pred):
    pred_ids  = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str  = [normalize_text(p) for p in processor.batch_decode(pred_ids,  skip_special_tokens=True)]
    label_str = [normalize_text(l) for l in processor.batch_decode(label_ids, skip_special_tokens=True)]
    return {'wer': 100 * metric.compute(predictions=pred_str, references=label_str)}

print('✅ Normalizer, collator and metrics ready')
print(f'  Normalizer test: "The Weather, Nice!" → "{normalize_text("The Weather, Nice!")}"')

✅ Normalizer, collator and metrics ready
  Normalizer test: "The Weather, Nice!" → "the weather nice"


## Step 8 — Experiment Configuration

In [11]:
START_MODEL = 'openai/whisper-tiny.en'
print(f'Starting from: {START_MODEL}')

EXPERIMENTS = [
    {
        'name': 'exp1_conservative',
        'learning_rate': 1e-5,
        'num_train_epochs': 3,
        'per_device_train_batch_size': 16,
        'warmup_steps': 10,
    },
    {
        'name': 'exp2_moderate',
        'learning_rate': 5e-5,
        'num_train_epochs': 5,
        'per_device_train_batch_size': 16,
        'warmup_steps': 20,
    },
    {
        'name': 'exp3_aggressive',
        'learning_rate': 1e-4,
        'num_train_epochs': 8,
        'per_device_train_batch_size': 16,
        'warmup_steps': 30,
    },
]

print(f'🧪 {len(EXPERIMENTS)} experiments configured:')
for exp in EXPERIMENTS:
    print(f"  → {exp['name']:<25} | LR: {exp['learning_rate']} | Epochs: {exp['num_train_epochs']}")

Starting from: openai/whisper-tiny.en
🧪 3 experiments configured:
  → exp1_conservative         | LR: 1e-05 | Epochs: 3
  → exp2_moderate             | LR: 5e-05 | Epochs: 5
  → exp3_aggressive           | LR: 0.0001 | Epochs: 8


## Step 9 — Run All Experiments

In [12]:
from transformers import WhisperForConditionalGeneration, Seq2SeqTrainer, Seq2SeqTrainingArguments
import json

all_results = []

for exp in EXPERIMENTS:
    print('\n' + '='*60)
    print(f"🚀 Running: {exp['name']}")
    print(f"   LR: {exp['learning_rate']} | Epochs: {exp['num_train_epochs']}")
    print('='*60)

    model = WhisperForConditionalGeneration.from_pretrained(START_MODEL)
    model.generation_config.forced_decoder_ids = None
    model.generation_config.suppress_tokens    = []

    output_dir = f"/content/combined-{exp['name']}"

    training_args = Seq2SeqTrainingArguments(
        output_dir=output_dir,
        per_device_train_batch_size=exp['per_device_train_batch_size'],
        per_device_eval_batch_size=8,
        gradient_accumulation_steps=1,
        learning_rate=exp['learning_rate'],
        warmup_steps=exp['warmup_steps'],
        num_train_epochs=exp['num_train_epochs'],
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='wer',
        greater_is_better=False,
        predict_with_generate=True,
        generation_max_length=225,
        logging_steps=5,
        report_to=['none'],
        fp16=True,
        dataloader_num_workers=2,
    )

    trainer = Seq2SeqTrainer(
        args=training_args,
        model=model,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        processing_class=processor.feature_extractor,
    )

    trainer.train()
    trainer.save_model(output_dir)
    processor.save_pretrained(output_dir)

    eval_results = trainer.evaluate()
    wer = eval_results.get('eval_wer', None)

    result = {
        'experiment': exp['name'],
        'learning_rate': exp['learning_rate'],
        'epochs': exp['num_train_epochs'],
        'wer': round(wer, 2) if wer else 'N/A',
        'model_path': output_dir
    }
    all_results.append(result)
    print(f"\n✅ {exp['name']} done | WER: {result['wer']}%")

with open('/content/experiment_results.json', 'w') as f:
    json.dump(all_results, f, indent=2)

print('\n🎉 All experiments complete!')


🚀 Running: exp1_conservative
   LR: 1e-05 | Epochs: 3


model.safetensors:   0%|          | 0.00/151M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/167 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

Epoch,Training Loss,Validation Loss,Wer
1,0.484879,0.388711,14.163614
2,0.298179,0.380480,14.529915
3,0.248080,0.386422,15.873016


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['proj_out.weight'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


✅ exp3_aggressive done | WER: 20.39%

🎉 All experiments complete!


## Step 10 — Compare Experiment Results

In [13]:
import json

with open('/content/experiment_results.json') as f:
    results = json.load(f)

results_df = pd.DataFrame(results)[['experiment', 'learning_rate', 'epochs', 'wer', 'model_path']]
results_df['wer'] = pd.to_numeric(results_df['wer'], errors='coerce')
results_df = results_df.sort_values('wer')

print('\n📊 Experiment Results (sorted by WER — lower is better):')
print('=' * 60)
print(results_df[['experiment', 'learning_rate', 'epochs', 'wer']].to_string(index=False))
print('=' * 60)

best = results_df.iloc[0]
print(f"\n🏆 Best: {best['experiment']} | WER: {best['wer']}%")


📊 Experiment Results (sorted by WER — lower is better):
       experiment  learning_rate  epochs   wer
exp1_conservative        0.00001       3 14.16
    exp2_moderate        0.00005       5 14.65
  exp3_aggressive        0.00010       8 20.39

🏆 Best: exp1_conservative | WER: 14.16%


## Step 11 — Final Comparison: Baseline vs Fine-Tuned

Evaluates both models on your held-out val set (your voice only).
Also tests on the external different-speaker clips.

In [14]:
from transformers import WhisperForConditionalGeneration, WhisperProcessor
import soundfile as sf, numpy as np, torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'

def run_inference(model, processor, eval_df, label='Model'):
    model.eval()
    model = model.to(device)
    preds, refs = [], []
    for _, row in eval_df.iterrows():
        audio, _ = sf.read(row['path'])
        audio = audio.astype(np.float32)
        if audio.ndim > 1: audio = audio.mean(axis=1)
        inputs = processor(audio, sampling_rate=16000, return_tensors='pt').input_features.to(device)
        with torch.no_grad():
            generated_ids = model.generate(inputs)
        pred = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
        preds.append(normalize_text(pred))
        refs.append(normalize_text(row['transcript']))
        if len(preds) % 20 == 0:
            print(f'  [{label}] {len(preds)}/{len(eval_df)} done...')
    wer = 100 * metric.compute(predictions=preds, references=refs)
    return wer, preds, refs

def print_comparison(baseline_wer, finetuned_wer, best_exp_name, val_size, set_label='val set'):
    improvement = baseline_wer - finetuned_wer
    if   improvement > 5:  verdict = '✅ Significant improvement'
    elif improvement > 0:  verdict = '🟡 Modest improvement'
    elif improvement == 0: verdict = '➡️  No change'
    else:                  verdict = '❌ Got worse — possible overfit'
    print('\n' + '='*60)
    print(f'📋 COMPARISON ({set_label}, normalized WER)')
    print('='*60)
    print(f'  Clips evaluated       : {val_size}')
    print(f'  Baseline Whisper Tiny : {baseline_wer:.2f}% WER')
    print(f'  Fine-tuned ({best_exp_name[:18]}) : {finetuned_wer:.2f}% WER')
    print(f'  Difference            : {improvement:+.2f}% WER')
    print(f'  Verdict               : {verdict}')
    print('='*60)

# ── Load baseline ─────────────────────────────────────────────────────────────
print('🔄 Loading baseline Whisper Tiny...')
baseline_model     = WhisperForConditionalGeneration.from_pretrained('openai/whisper-tiny.en')
baseline_processor = WhisperProcessor.from_pretrained('openai/whisper-tiny.en')

# ── Load best fine-tuned ──────────────────────────────────────────────────────
best_model_path = results_df.iloc[0]['model_path']
best_exp_name   = results_df.iloc[0]['experiment']
print(f'🔄 Loading best fine-tuned model: {best_exp_name}...')
finetuned_model     = WhisperForConditionalGeneration.from_pretrained(best_model_path)
finetuned_processor = WhisperProcessor.from_pretrained(best_model_path)

# ── Eval on val set ───────────────────────────────────────────────────────────
print('\n── Val Set Evaluation ──')
baseline_wer_val,  baseline_preds_val,  refs_val = run_inference(baseline_model,   baseline_processor,  val_df.reset_index(),  'Baseline')
finetuned_wer_val, finetuned_preds_val, _        = run_inference(finetuned_model,  finetuned_processor, val_df.reset_index(),  'Fine-tuned')
print_comparison(baseline_wer_val, finetuned_wer_val, best_exp_name, len(val_df), 'your voice — val set')

print('\n🔍 Sample val predictions (first 5):')
for i, (_, row) in enumerate(val_df.reset_index().head(5).iterrows()):
    print(f'\n  REF      : "{normalize_text(row["transcript"])}"')
    print(f'  BASELINE : "{baseline_preds_val[i]}"')
    print(f'  FINETUNED: "{finetuned_preds_val[i]}"')

🔄 Loading baseline Whisper Tiny...


Loading weights:   0%|          | 0/167 [00:00<?, ?it/s]

🔄 Loading best fine-tuned model: exp1_conservative...


Loading weights:   0%|          | 0/167 [00:00<?, ?it/s]


── Val Set Evaluation ──
  [Baseline] 20/50 done...
  [Baseline] 40/50 done...
  [Fine-tuned] 20/50 done...
  [Fine-tuned] 40/50 done...

📋 COMPARISON (your voice — val set, normalized WER)
  Clips evaluated       : 50
  Baseline Whisper Tiny : 18.32% WER
  Fine-tuned (exp1_conservative) : 14.16% WER
  Difference            : +4.15% WER
  Verdict               : 🟡 Modest improvement

🔍 Sample val predictions (first 5):

  REF      : "a simple definition is machine learning is about building algorithms that learn patterns from data and use those patterns to make predictions"
  BASELINE : "a simple definition is machine learning is about building algorithms that learn patterns from data and use those patterns to make predictions"
  FINETUNED: "a simple definition is machine learning is about building algorithms that learn patterns from data and use those patterns to make predictions"

  REF      : "historical junction its interesting to note fft wasnt invented by one person"
  BASELINE 

## Step 12 — Test on Different Speaker Clips

In [15]:
# ── Eval on external test clips (different voice) ────────────────────────────
print('\n📁 Upload test clips zip (clips/ folder + metadata.csv, different speaker)...')
test_uploaded = files.upload()
test_zip = list(test_uploaded.keys())[0]
with zipfile.ZipFile(test_zip, 'r') as z:
    z.extractall('/content/test_clips/')

test_meta = pd.read_csv('/content/test_clips/metadata.csv')
test_meta = test_meta.dropna(subset=['file_name', 'transcript'])
test_meta['path'] = test_meta['file_name'].apply(
    lambda f: os.path.join('/content/test_clips/clips', f)
)
test_meta = test_meta[test_meta['path'].apply(os.path.exists)].reset_index(drop=True)
print(f'✅ {len(test_meta)} test clips loaded')

baseline_wer_test,  baseline_preds_test,  refs_test = run_inference(baseline_model,  baseline_processor,  test_meta, 'Baseline')
finetuned_wer_test, finetuned_preds_test, _         = run_inference(finetuned_model, finetuned_processor, test_meta, 'Fine-tuned')
print_comparison(baseline_wer_test, finetuned_wer_test, best_exp_name, len(test_meta), 'different speaker — test clips')

print('\n🔍 All test clip predictions:')
for i, (_, row) in enumerate(test_meta.iterrows()):
    print(f'\n  REF      : "{normalize_text(row["transcript"])}"')
    print(f'  BASELINE : "{baseline_preds_test[i]}"')
    print(f'  FINETUNED: "{finetuned_preds_test[i]}"')


📁 Upload test clips zip (clips/ folder + metadata.csv, different speaker)...


KeyboardInterrupt: 

## Step 13 — Download Best Model

In [ ]:
import shutil
from google.colab import files

best_path = results_df.iloc[0]['model_path']
zip_path  = '/content/best_model.zip'
shutil.make_archive('/content/best_model', 'zip', best_path)
print(f'📦 Zipping {best_path}...')
files.download(zip_path)

In [ ]:
import os, glob, json

checkpoints = glob.glob('/content/combined-*', recursive=False)
print('Checkpoints found:')
for c in checkpoints: print(f'  {c}')

if os.path.exists('/content/experiment_results.json'):
    with open('/content/experiment_results.json') as f:
        print('\nexperiment_results.json:')
        print(json.dumps(json.load(f), indent=2))
else:
    print('❌ No results found — runtime was fully reset')